# Experiment Matrix: Men's LR Feature Expansion & Interactions

Systematically tests feature additions and interaction terms for men's logistic regression.

**Validation protocol**: Every config is evaluated on both:
1. Holdout (train <2022, test >=2022) — quick sanity check
2. Full LOSO CV — the real test

**Acceptance criteria**:
- EXP-1: LOSO improves over baseline by >=0.001 AND holdout doesn't degrade
- EXP-2: LOSO improves AND coefficients stable (same sign) across >=80% of folds

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
import sys

project_root = Path.cwd().parent
sys.path.insert(0, str(project_root))

from src.data_loader import load_men_data, load_women_data
from src.feature_engineering import *
from src.model import brier_score
from src.model_logreg import (
    train_logreg, predict_logreg, logreg_cv, logreg_holdout_eval,
    print_logreg_coefficients, run_experiment, add_interactions,
    MINIMAL_FEATURES, MINIMAL_FEATURES_MEN, EXPANDED_FEATURES_MEN
)

## Load Data & Build Features

In [ ]:
data_dir = f'{project_root}/data'

# Men
m_data = load_men_data(data_dir)
m_elo = compute_elo_ratings(m_data['MComSsn'])
m_seeds = parse_seeds(m_data['MTrnySeeds'])
m_stats = compute_season_stats(m_data['MDetSsn'])
massey = compute_massey_features(m_data['MOrdinals'])
m_conf = compute_conference_strength(m_data['MConf'], m_elo)
m_eff = compute_efficiency(m_data['MDetSsn'])
m_features = build_team_features(m_elo, m_seeds, m_stats, massey, m_conf, efficiency_df=m_eff)

# Build matchup matrix
m_matchups = build_matchup_matrix(m_data['MDetTrny'], m_features)

print(f'Matchups shape: {m_matchups.shape}')
print(f'Seasons: {sorted(m_matchups["Season"].unique())}')
print(f'Available diff features: {[c for c in m_matchups.columns if c.endswith("_diff")]}')

## EXP-1: Men's LR Feature Expansion

Test adding Off_Eff_diff and Win_pct_diff to the 3-feature baseline.

In [ ]:
all_results = []

# EXP-1a: Baseline (3 features, C=1.0)
print('=== EXP-1a: Baseline (Elo, Seed, POM) ===')
res = run_experiment(m_matchups, MINIMAL_FEATURES_MEN, [1.0], 'EXP-1a: baseline')
all_results.extend(res)
print()

In [ ]:
# EXP-1b: Add Off_Eff
print('=== EXP-1b: + Off_Eff ===')
feat_1b = ['Elo_diff', 'SeedNum_diff', 'Rank_POM_diff', 'Off_Eff_diff']
res = run_experiment(m_matchups, feat_1b, [1.0], 'EXP-1b: +Off_Eff')
all_results.extend(res)
print()

# EXP-1c: Add Win_pct
print('=== EXP-1c: + Win_pct ===')
feat_1c = ['Elo_diff', 'SeedNum_diff', 'Rank_POM_diff', 'Win_pct_diff']
res = run_experiment(m_matchups, feat_1c, [1.0], 'EXP-1c: +Win_pct')
all_results.extend(res)
print()

# EXP-1d: Add both
print('=== EXP-1d: + Off_Eff + Win_pct ===')
res = run_experiment(m_matchups, EXPANDED_FEATURES_MEN, [1.0], 'EXP-1d: +Off_Eff+Win_pct')
all_results.extend(res)

In [ ]:
# EXP-1e: C tuning for the expanded feature set
print('=== EXP-1e: C tuning for expanded features ===')
C_grid = [0.01, 0.05, 0.1, 0.5, 1.0]
res = run_experiment(m_matchups, EXPANDED_FEATURES_MEN, C_grid, 'EXP-1e: expanded+C')
all_results.extend(res)

In [ ]:
# Also tune C for the baseline 3-feature set for fair comparison
print('=== EXP-1a*: C tuning for baseline ===')
res = run_experiment(m_matchups, MINIMAL_FEATURES_MEN, C_grid, 'EXP-1a*: baseline+C')
all_results.extend(res)

## EXP-1 Results Summary

In [ ]:
df_results = pd.DataFrame(all_results)
print(df_results[['label', 'C', 'holdout_brier', 'loso_brier', 'coef_stability']].to_string(index=False))

# Find best config
best = df_results.loc[df_results['loso_brier'].idxmin()]
print(f'\nBest config by LOSO: {best["label"]} C={best["C"]} -> LOSO={best["loso_brier"]:.5f}, Holdout={best["holdout_brier"]:.5f}')

## EXP-2: Men's Interaction Terms

Built on top of the best feature set from EXP-1. Tests targeted interactions.

In [ ]:
# Use the best C from EXP-1 results (or default to the best LOSO C)
# We'll test interactions with both C=1.0 and the best C from EXP-1e
best_exp1 = df_results[df_results['label'].str.startswith('EXP-1e')].sort_values('loso_brier').iloc[0]
best_C = best_exp1['C']
print(f'Using best C from EXP-1e: {best_C}')

exp2_results = []

# EXP-2a: Elo * Seed
print('\n=== EXP-2a: Elo x Seed ===')
res = run_experiment(m_matchups, EXPANDED_FEATURES_MEN, [best_C],
                     'EXP-2a: +Elo*Seed',
                     interaction_pairs=[('Elo_diff', 'SeedNum_diff')])
exp2_results.extend(res)

# EXP-2b: Elo * POM
print('\n=== EXP-2b: Elo x POM ===')
res = run_experiment(m_matchups, EXPANDED_FEATURES_MEN, [best_C],
                     'EXP-2b: +Elo*POM',
                     interaction_pairs=[('Elo_diff', 'Rank_POM_diff')])
exp2_results.extend(res)

# EXP-2c: Both
print('\n=== EXP-2c: Elo*Seed + Elo*POM ===')
res = run_experiment(m_matchups, EXPANDED_FEATURES_MEN, [best_C],
                     'EXP-2c: +Elo*Seed+Elo*POM',
                     interaction_pairs=[('Elo_diff', 'SeedNum_diff'),
                                       ('Elo_diff', 'Rank_POM_diff')])
exp2_results.extend(res)

# EXP-2d: All three interactions
print('\n=== EXP-2d: + Off_Eff*Win_pct ===')
res = run_experiment(m_matchups, EXPANDED_FEATURES_MEN, [best_C],
                     'EXP-2d: +3 interactions',
                     interaction_pairs=[('Elo_diff', 'SeedNum_diff'),
                                       ('Elo_diff', 'Rank_POM_diff'),
                                       ('Off_Eff_diff', 'Win_pct_diff')])
exp2_results.extend(res)

In [ ]:
# EXP-2 with C tuning for best interaction set
print('=== EXP-2e: C sweep for best interaction config ===')
# Find best interaction config
df_exp2 = pd.DataFrame(exp2_results)
best_int = df_exp2.loc[df_exp2['loso_brier'].idxmin()]
print(f'Best interaction config: {best_int["label"]}')

# Determine which interaction pairs to use based on best label
int_map = {
    'EXP-2a: +Elo*Seed': [('Elo_diff', 'SeedNum_diff')],
    'EXP-2b: +Elo*POM': [('Elo_diff', 'Rank_POM_diff')],
    'EXP-2c: +Elo*Seed+Elo*POM': [('Elo_diff', 'SeedNum_diff'), ('Elo_diff', 'Rank_POM_diff')],
    'EXP-2d: +3 interactions': [('Elo_diff', 'SeedNum_diff'), ('Elo_diff', 'Rank_POM_diff'), ('Off_Eff_diff', 'Win_pct_diff')],
}
best_int_pairs = int_map[best_int['label']]

res = run_experiment(m_matchups, EXPANDED_FEATURES_MEN, C_grid,
                     'EXP-2e: best_int+C',
                     interaction_pairs=best_int_pairs)
exp2_results.extend(res)

## EXP-2 Results Summary

In [ ]:
df_exp2 = pd.DataFrame(exp2_results)
print(df_exp2[['label', 'C', 'holdout_brier', 'loso_brier', 'coef_stability']].to_string(index=False))

## Combined Results: All Experiments

In [ ]:
df_all = pd.DataFrame(all_results + exp2_results)

# Sort by LOSO brier
df_sorted = df_all.sort_values('loso_brier')
print('=== ALL RESULTS (sorted by LOSO Brier) ===')
print(df_sorted[['label', 'C', 'holdout_brier', 'loso_brier', 'coef_stability']].to_string(index=False))

print(f'\n--- Top 5 by LOSO ---')
for _, row in df_sorted.head(5).iterrows():
    print(f"  {row['label']:35s} C={row['C']:<6.3f} LOSO={row['loso_brier']:.5f} "
          f"Holdout={row['holdout_brier']:.5f} Stability={row['coef_stability']:.1%}")

## Decision Criteria

Check the results above against acceptance criteria:
- **EXP-1 accept if**: LOSO improves over baseline (EXP-1a) by >=0.001 AND holdout doesn't degrade
- **EXP-2 accept if**: LOSO improves AND coefficient stability >=80%

Use the best accepted config to update `EXPANDED_FEATURES_MEN` and the submission notebook.